# 02 — OpenAI Streaming: Examples

> Master the OpenAI streaming API — context managers, tool call detection, usage tracking.

**What you'll build:**
1. Text streaming with context manager
2. Usage statistics during streaming
3. Tool call detection mid-stream
4. Streaming + structured output (JSON)
5. Finish reason handler

In [ ]:
import os, time, json
from openai import OpenAI
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

load_dotenv()
client = OpenAI()
console = Console()
print('✅ Setup complete')

---
## Part 1: Context Manager Streaming

In [ ]:
# Clean streaming using the .stream() context manager
print("🌊 Streaming with context manager:\n")

with client.chat.completions.stream(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "List 3 benefits of streaming LLM responses, one per line."}]
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

    # After loop — get the fully assembled response
    final = stream.get_final_completion()

rprint(f"\n\n[bold]Final object:[/bold]")
rprint(f"   finish_reason: {final.choices[0].finish_reason}")
rprint(f"   content length: {len(final.choices[0].message.content)} chars")

---
## Part 2: Usage Statistics During Streaming

In [ ]:
PRICING = {"gpt-4o-mini": (0.15e-6, 0.60e-6)}

def stream_with_cost(prompt: str, model: str = "gpt-4o-mini") -> dict:
    full_text = ""
    usage = None

    stream = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
        stream_options={"include_usage": True}  # Request token counts
    )

    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            content = chunk.choices[0].delta.content
            full_text += content
            print(content, end="", flush=True)
        if chunk.usage:
            usage = chunk.usage

    print()
    pi, po = PRICING.get(model, (5e-6, 15e-6))
    cost = usage.prompt_tokens * pi + usage.completion_tokens * po if usage else 0

    return {
        "text": full_text,
        "input_tokens": usage.prompt_tokens if usage else None,
        "output_tokens": usage.completion_tokens if usage else None,
        "cost_usd": cost,
    }


print("\n📤 Streaming with cost tracking:\n")
result = stream_with_cost("Explain what a token is in the context of LLMs in 2 sentences.")

rprint(f"\n[bold]Cost breakdown:[/bold]")
rprint(f"   Input:  {result['input_tokens']} tokens")
rprint(f"   Output: {result['output_tokens']} tokens")
rprint(f"   Cost:   [green]${result['cost_usd']:.6f}[/green]")

---
## Part 3: Tool Call Detection Mid-Stream

In [ ]:
import json

# Define a tool for the model to call
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "The city name"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}
            },
            "required": ["city"]
        }
    }
}]


def stream_with_tool_detection(user_message: str) -> dict:
    """Stream and detect tool calls as they arrive."""
    full_text = ""
    tool_calls_acc = {}  # {index: {id, name, arguments}}
    finish_reason = None

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": user_message}],
        tools=TOOLS,
        stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        finish_reason = chunk.choices[0].finish_reason or finish_reason

        if delta.content:
            full_text += delta.content
            print(delta.content, end="", flush=True)

        # Accumulate tool call deltas
        if delta.tool_calls:
            for tc in delta.tool_calls:
                idx = tc.index
                if idx not in tool_calls_acc:
                    tool_calls_acc[idx] = {"id": "", "name": "", "arguments": ""}
                if tc.id:                        tool_calls_acc[idx]["id"]        += tc.id
                if tc.function.name:             tool_calls_acc[idx]["name"]      += tc.function.name
                if tc.function.arguments:        tool_calls_acc[idx]["arguments"] += tc.function.arguments

    # Parse tool call arguments
    tool_calls_parsed = []
    for idx, tc in tool_calls_acc.items():
        try:
            tc["args"] = json.loads(tc["arguments"])
        except:
            tc["args"] = {}
        tool_calls_parsed.append(tc)

    return {"finish_reason": finish_reason, "text": full_text, "tool_calls": tool_calls_parsed}


# Test: message that should trigger a tool call
print("📤 Request: 'What's the weather in Tokyo?'\n")
result = stream_with_tool_detection("What's the weather in Tokyo?")

rprint(f"\n[bold]Result:[/bold]")
rprint(f"   finish_reason: {result['finish_reason']}")
if result['tool_calls']:
    for tc in result['tool_calls']:
        rprint(f"   Tool called: [green]{tc['name']}[/green]")
        rprint(f"   Arguments:   {tc['args']}")

---
## Part 4: Finish Reason Handler

In [ ]:
FINISH_REASON_ACTIONS = {
    "stop":           lambda r: rprint("[green]✅ Normal completion[/green]"),
    "tool_calls":     lambda r: rprint(f"[yellow]🔧 Tool call needed: {r.get('tool_calls')}[/yellow]"),
    "length":         lambda r: rprint("[red]✂️ Truncated — increase max_tokens[/red]"),
    "content_filter": lambda r: rprint("[red]🚫 Content filtered — rephrase the request[/red]"),
}

def handle_stream(prompt: str, max_tokens: int = 500) -> dict:
    full_text = ""
    tool_calls_acc = {}
    finish_reason = None

    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        tools=TOOLS,
        max_tokens=max_tokens,
        stream=True
    )

    for chunk in stream:
        delta = chunk.choices[0].delta
        if delta.content:
            full_text += delta.content
            print(delta.content, end="", flush=True)
        if delta.tool_calls:
            for tc in delta.tool_calls:
                idx = tc.index
                if idx not in tool_calls_acc:
                    tool_calls_acc[idx] = {"name": "", "arguments": ""}
                if tc.function.name:      tool_calls_acc[idx]["name"]      += tc.function.name
                if tc.function.arguments: tool_calls_acc[idx]["arguments"] += tc.function.arguments
        finish_reason = chunk.choices[0].finish_reason or finish_reason

    print()
    result = {"text": full_text, "tool_calls": list(tool_calls_acc.values()), "finish_reason": finish_reason}

    action = FINISH_REASON_ACTIONS.get(finish_reason, lambda r: rprint(f"Unknown: {finish_reason}"))
    action(result)
    return result


# Test cases
test_cases = [
    ("Say hello in one word.",           500, "Normal stop"),
    ("Count from 1 to 100 in full.",     10,  "Truncated (max_tokens=10)"),
    ("What's the weather in London?",    500, "Tool call"),
]

for prompt, max_tok, label in test_cases:
    rprint(f"\n[cyan]── {label} ──[/cyan]")
    rprint(f"Prompt: {prompt}")
    handle_stream(prompt, max_tokens=max_tok)

---
## Summary

| Feature | How to use |
|---|---|
| Context manager | `with client.chat.completions.stream(...) as s:` |
| Text stream | `for text in s.text_stream:` |
| Usage stats | `stream_options={"include_usage": True}` |
| Tool call detection | Accumulate `delta.tool_calls[i].function.arguments` |
| Finish reason | Check `chunk.choices[0].finish_reason` on each chunk |

**Next:** `03_streaming_across_providers` — streaming with Anthropic, Gemini, and LiteLLM.